In [ ]:
!pip install -q unsloth trl peft accelerate bitsandbytes datasets transformers pandas

In [ ]:
import torch
print("GPU is available: ", torch.cuda.is_available())
print("Device is: ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')

GPU is available:  True
Device is:  Tesla T4


In [ ]:
import pandas as pd
import json
from unsloth import FastLanguageModel

with open('healthcare_data2.json', 'r', encoding='utf-8') as f:
    data = json.load(f)


records = []
for item in data:
    user_text = item['prompt'][0]['content'] if isinstance(item['prompt'], list) else item['prompt']['content']
    assistant_text = item['completion'][0]['content'] if isinstance(item['completion'], list) else item['completion']['content']

    records.append({
        "user": user_text,
        "assistant": assistant_text
    })

df = pd.DataFrame(records)

def formatting_prompts_func(examples):
    convos = []
    for user, assistant in zip(examples['user'], examples['assistant']):
        text = f"<|start_header_id|>user<|end_header_id|>\n\n{user}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{assistant}<|eot_id|>"
        convos.append(text)
    return {"text": convos}

from datasets import Dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.map(formatting_prompts_func, batched=True)

print(dataset[0]['text'][:200])

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

<|start_header_id|>user<|end_header_id|>

I am a 32-year-old female experiencing shortness of breath for 2 days after exercise. What could be the cause and what should I do?<|eot_id|><|start_header_id


In [ ]:
# install LLM in 4-bit format
from unsloth import FastLanguageModel
import torch

max_seq = 2048
dtype = None
load_in_4_bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit",
    max_seq_length = max_seq, # Choose any for long context!
    load_in_4bit = load_in_4_bit,  # 4 bit quantization to reduce memory
    dtype = dtype,
)

==((====))==  Unsloth 2026.4.4: Fast Mistral patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

Testing installed LLM

In [ ]:
def test_model(prompt_text):
    messages = [
        {"role": "user", "content": prompt_text},
    ]
    # tokenization
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")

    # generation
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=1048,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    # decoded answers
    response = tokenizer.decode(outputs[0][len(inputs[0]):], skip_special_tokens=True)
    return response

# Quick start
print("\n", "Модель готова к тестам!", "\n", "Model is ready for tests!")


 Модель готова к тестам! 
 Model is ready for tests!


In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Test prompt
messages = [
    {"role": "user", "content": "I am a 60-year-old with hypertension experiencing bloating, rash, abdominal pain for 2 days at night. What could be the cause and what should I do?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

# Decode and print
response = tokenizer.batch_decode(outputs)[0]
print(response)

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

<|user|> I am a 60-year-old with hypertension experiencing bloating, rash, abdominal pain for 2 days at night. What could be the cause and what should I do?<|end|><|assistant|> Given your symptoms of bloating, rash, and abdominal pain, particularly at night, along with your history of hypertension, there are several potential causes. It's important to consider both gastrointestinal and allergic reactions, as well as the possibility of stress-related factors.


1. **Gastrointestinal Causes**: These could include conditions such as gastroesophageal reflux disease (GERD), peptic ulcers, or gastritis, which might be exacerbated by stress or certain foods.

2. **Allergic Reactions**: The rash could be a sign of an allergic reaction, possibly to a medication or food.

3. **Stress**: Stress can exacerbate symptoms of hypertension and can also cause or worsen gastrointest0inal symptoms.


Given your age and existing hypertension, it's important to approach this carefully. I recommend the follo

In [ ]:
# add LoRa adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha=128,
    lora_dropout=0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None
)

Unsloth 2026.4.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
# start LLM training
from transformers import TrainingArguments
from trl import SFTTrainer
import torch

max_seq = 2048

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text", # Имя колонки, куда мы положили отформатированный текст
    max_seq_length=max_seq,
    dataset_num_proc=2,
    packing=False, # Можно поставить True для ускорения, если данные короткие, но False безопаснее для начала
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=25,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        save_strategy="epoch",
        save_total_limit=2,
        dataloader_pin_memory=False,
        report_to="none",
    ),
)

trainer.train()

Unsloth: Switching to float32 training since model cannot work with float16


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10000 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 3 | Total steps = 3,750
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 119,537,664 of 3,940,617,216 (3.03% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
25,0.667357
50,0.141878
75,0.136679
100,0.132178
125,0.133553
150,0.129397
175,0.130724
200,0.131816
225,0.130440
250,0.130629


TrainOutput(global_step=3750, training_loss=0.060402154286702474, metrics={'train_runtime': 7915.7762, 'train_samples_per_second': 3.79, 'train_steps_per_second': 0.474, 'total_flos': 1.2657448974388838e+17, 'train_loss': 0.060402154286702474, 'epoch': 3.0})

In [ ]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

# Test prompt
messages = [
    {"role": "user", "content": "I am a 32-year-old female experiencing shortness of breath for 2 days after exercise. What could be the cause and what should I do?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

# Generate response
outputs = model.generate(
    input_ids=inputs,
    max_new_tokens=256,
    use_cache=True,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

# Decode and print
response = tokenizer.batch_decode(outputs)[0]
print(response)

Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<|user|> I am a 32-year-old female experiencing shortness of breath for 2 days after exercise. What could be the cause and what should I do?<|end|><|assistant|> Based on your description, common possibilities include seasonal allergies. This is not a diagnosis. Consider supportive care: consider an antacid trial. If you develop any red-flag symptoms such as stiff neck with fever, seek urgent medical attention. For persistent or worsening symptoms, consult a licensed clinician who can examine you and order appropriate tests. For persistent or worsening symptoms at rest, seek urgent medical attention. For persistent pain, consult a licensed clinician who can examine you and order appropriate tests. For common symptoms such as seasonal allergies, consider supportive care: monitor symptoms and seek care if they worsen. If you develop any high-risk symptoms such as confusion, seek urgent medical attention. For persistent or worsening symptoms at rest, consult a licensed clinician who can ex